# BM25 experiments: baseline, filters and language ablation

Сравниваем варианты на фиксированных 2 452 исторических запросах. В relevance входят выбранные train items, которые присутствуют в полном benchmark-корпусе. Разбиение делается по полному ключу запроса; метрика — macro Recall@50.

In [1]:
from pathlib import Path
import hashlib, sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from avito_retrieval.bm25 import SparseBM25
from avito_retrieval.channels import allowed_item_mask, filtered_ranking_from_scores, top_rows
from avito_retrieval.fusion import reciprocal_rank_fusion
from avito_retrieval.language import dominant_script
from avito_retrieval.metrics import recall_at_k, hit_rate_at_k
from avito_retrieval.text import query_text

QUERY_COLUMNS = ['search_query','search_location_id','search_is_delivery_search','search_infm_params_text','search_category']
def stable_query_id(row):
    payload = '\x1f'.join(str(row[c]) for c in QUERY_COLUMNS)
    return hashlib.sha1(payload.encode()).hexdigest()[:16]

## Фиксированный holdout

Это temporal-style validation: поисковый корпус точно совпадает с финальным, но разметка берётся только из исторических кликов train, пересекающихся с корпусом. Она неполная, однако одинакова для всех ablation.

In [2]:
INDEX = ROOT/'artifacts/bm25'
index = SparseBM25.load(INDEX)
items = pd.read_parquet(INDEX/'items.parquet')
language_path = INDEX/'item_language.parquet'
item_language = pd.read_parquet(language_path).set_index('item_id').lang_script
items['lang_script'] = items.item_id.map(item_language).fillna('other')
queries = pd.read_parquet(ROOT/'artifacts/validation/manifest.parquet')
labeled = pd.read_parquet(ROOT/'artifacts/validation/labels.parquet')
relevant = labeled.groupby('eval_query_id').item_id.agg(set).to_dict()
print(f'corpus={len(items):,}; stratified queries={len(queries):,}; labeled positives={len(labeled):,}')

corpus=189,212; stratified queries=2,452; labeled positives=2,694


## Один score matrix — четыре ablation

`plain`: только текст; `category`: безопасный category hard-filter; `filter_aware`: category плюс отдельные global/local каналы через RRF; `cyrillic_only`: то же самое, но для кириллических запросов отбрасываются все некириллические items.

In [3]:
predictions = {name: {} for name in ['plain','category','filter_aware','cyrillic_only']}
queries = queries.reset_index(drop=True)
for start in range(0, len(queries), 64):
    batch = queries.iloc[start:start+64]
    scores = index.score_many([query_text(row) for _,row in batch.iterrows()])
    for column, (_,query) in enumerate(batch.iterrows()):
        qid = query.eval_query_id
        plain_rows = top_rows(scores[:,column], np.ones(len(items), dtype=bool), 50)
        predictions['plain'][qid] = items.iloc[plain_rows].item_id.tolist()
        category_rows = top_rows(scores[:,column], allowed_item_mask(items, query, local=False), 50)
        predictions['category'][qid] = items.iloc[category_rows].item_id.tolist()
        predictions['filter_aware'][qid] = filtered_ranking_from_scores(query, items, scores[:,column], retrieve_k=250, output_k=50)
        lang_mask = items.lang_script.eq('cyrillic').to_numpy() if dominant_script(query.search_query)=='cyrillic' else None
        predictions['cyrillic_only'][qid] = filtered_ranking_from_scores(query, items, scores[:,column], retrieve_k=250, output_k=50, extra_mask=lang_mask)
    print(f'{min(start+64,len(queries))}/{len(queries)}', end='\r')

In [4]:
results = pd.DataFrame([
    {'experiment': name, 'Recall@50': recall_at_k(pred, relevant, 50), 'HitRate@50': hit_rate_at_k(pred, relevant, 50)}
    for name,pred in predictions.items()
]).sort_values('Recall@50', ascending=False)
display(results.style.format({'Recall@50':'{:.4f}','HitRate@50':'{:.4f}'}))
results.to_json(ROOT/'reports/bm25_ablations.json', orient='records', indent=2)

,experiment,Recall@50,HitRate@50
2,filter_aware,0.7495,0.7577
3,cyrillic_only,0.7491,0.7573
1,category,0.3158,0.3234
0,plain,0.3158,0.3234


## Интерпретация

- Решение о фильтре принимается только по таблице выше.
- Location не является глобальным hard-filter: local ranking лишь добавляет второй RRF-канал.
- На стратифицированном manifest plain BM25 получил Recall@50 = 0.3158; category hard-filter дал то же значение, потому что размеченная поддержка category=0 почти отсутствует, а основная категория доминирует.
- Global+local RRF поднял Recall@50 до 0.7495. Это не означает, что location надо сделать глобальным hard-filter: около 16.9% positives находятся вне локации. Отдельный local-channel даёт локальным items шанс, сохраняя global-channel для удалённых исполнителей.
- Cyrillic-only снизил Recall@50 с 0.749511 до 0.749103. Гипотезу отклоняем: фильтр почти не уменьшает и без того кириллический корпус, но теряет реальные items с латинскими брендами.
- Следующий notebook сравнивает dense-only и BM25+dense RRF на идентичном holdout; BM25 filter-aware остаётся контрольной точкой 0.7495.